# Missing Value Imputation
**Improvements:**
- Fixed `FutureWarning`/`SettingWithCopyWarning` — all slice assignments replaced with `df.loc[idx, col] = ...`
- Consolidated 3 repetitive mode-imputation functions into 1 hierarchical function
- Added imputation quality validation (before/after stats)
- Added rationale comments for every imputation strategy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

df = pd.read_csv('gurgaon_properties_outlier_treated.csv')
print(f"Shape: {df.shape}")
print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum() > 0])

## Area Imputation
When `built_up_area` is missing, estimate from `super_built_up_area` and/or `carpet_area` using median ratios derived from rows where all three are present.

In [ ]:
# Calculate imputation ratios from complete rows only
complete = df.dropna(subset=['super_built_up_area','built_up_area','carpet_area'])
sbu_ratio    = (complete['super_built_up_area'] / complete['built_up_area']).median()
carpet_ratio = (complete['carpet_area']         / complete['built_up_area']).median()
print(f"Rows with all 3 area types: {len(complete)}")
print(f"super_built_up / built_up ratio (median): {sbu_ratio:.4f}")
print(f"carpet / built_up ratio (median)         : {carpet_ratio:.4f}")

In [ ]:
# Strategy 1: both super_built_up AND carpet present, built_up missing
mask1 = df['super_built_up_area'].notna() & df['built_up_area'].isna() & df['carpet_area'].notna()
idx1  = df[mask1].index
# FIX: use df.loc[] directly — avoids SettingWithCopyWarning and FutureWarning
df.loc[idx1, 'built_up_area'] = round(
    (df.loc[idx1, 'super_built_up_area'] / sbu_ratio +
     df.loc[idx1, 'carpet_area']         / carpet_ratio) / 2
)
print(f"Strategy 1 imputed: {mask1.sum()} rows")

In [ ]:
# Strategy 2: only super_built_up present, built_up missing
mask2 = df['super_built_up_area'].notna() & df['built_up_area'].isna() & df['carpet_area'].isna()
idx2  = df[mask2].index
df.loc[idx2, 'built_up_area'] = round(df.loc[idx2, 'super_built_up_area'] / sbu_ratio)
print(f"Strategy 2 imputed: {mask2.sum()} rows")

In [ ]:
# Strategy 3: only carpet present, built_up missing
mask3 = df['super_built_up_area'].isna() & df['built_up_area'].isna() & df['carpet_area'].notna()
idx3  = df[mask3].index
df.loc[idx3, 'built_up_area'] = round(df.loc[idx3, 'carpet_area'] / carpet_ratio)
print(f"Strategy 3 imputed: {mask3.sum()} rows")

In [ ]:
# Anomaly correction: low area but suspiciously high price
# These rows likely have sqft recorded as sqm — replace built_up_area with raw 'area' column
anom_idx = df[(df['built_up_area'] < 2000) & (df['price'] > 2.5)].index
df.loc[anom_idx, 'built_up_area'] = df.loc[anom_idx, 'area']
print(f"Anomalies corrected (area replaced): {len(anom_idx)}")

# Drop intermediate area columns no longer needed
df.drop(columns=['area','areaWithType','super_built_up_area','carpet_area','area_room_ratio'],
        inplace=True, errors='ignore')
print(f"Shape after area cleanup: {df.shape}")

## Floor Number Imputation
17 missing values, mostly houses. Use property-type median.

In [ ]:
print(f"Missing floorNum: {df['floorNum'].isna().sum()}")
median_floor_house = df[df['property_type'] == 'house']['floorNum'].median()
median_floor_flat  = df[df['property_type'] == 'flat']['floorNum'].median()
print(f"Median floor — house: {median_floor_house}, flat: {median_floor_flat}")

# FIX: use loc-based assignment instead of df['col'].fillna(inplace=True) on slice
house_na = df[(df['property_type'] == 'house') & df['floorNum'].isna()].index
flat_na  = df[(df['property_type'] == 'flat')  & df['floorNum'].isna()].index
df.loc[house_na, 'floorNum'] = median_floor_house
df.loc[flat_na,  'floorNum'] = median_floor_flat
print(f"Missing floorNum after imputation: {df['floorNum'].isna().sum()}")

## Facing Column
~28% missing. Dropping is appropriate as imputation with mode would be uninformative and the feature has low predictive power.

In [ ]:
print(f"Facing missing: {df['facing'].isna().sum()} ({df['facing'].isna().mean()*100:.1f}%)")
df.drop(columns=['facing'], inplace=True)

## Society — Drop Single Missing Row

In [ ]:
missing_society = df[df['society'].isna()].index
print(f"Missing society rows: {list(missing_society)}")
df.drop(index=missing_society, inplace=True)

## Age/Possession Imputation
Original code had 3 separate `.apply()` functions doing the same thing at different hierarchy levels.

**Fix:** Single hierarchical function — tries sector+type, then sector, then property_type, then global mode.

In [ ]:
print(f"'Undefined' agePossession before: {(df['agePossession'] == 'Undefined').sum()}")

def impute_age_possession(df):
    result = df['agePossession'].copy()
    undefined_idx = df[df['agePossession'] == 'Undefined'].index

    for idx in undefined_idx:
        sector = df.loc[idx, 'sector']
        ptype  = df.loc[idx, 'property_type']

        # Level 1: same sector + same property_type
        candidates = df[(df['sector'] == sector) &
                        (df['property_type'] == ptype) &
                        (df['agePossession'] != 'Undefined')]['agePossession']
        if not candidates.empty:
            result[idx] = candidates.mode()[0]; continue

        # Level 2: same sector only
        candidates = df[(df['sector'] == sector) &
                        (df['agePossession'] != 'Undefined')]['agePossession']
        if not candidates.empty:
            result[idx] = candidates.mode()[0]; continue

        # Level 3: same property_type only
        candidates = df[(df['property_type'] == ptype) &
                        (df['agePossession'] != 'Undefined')]['agePossession']
        if not candidates.empty:
            result[idx] = candidates.mode()[0]; continue

        # Level 4: global mode fallback
        result[idx] = df[df['agePossession'] != 'Undefined']['agePossession'].mode()[0]

    return result

df['agePossession'] = impute_age_possession(df)
print(f"'Undefined' agePossession after : {(df['agePossession'] == 'Undefined').sum()}")

## Validation — Before vs After

In [ ]:
print("=== Missing values after all imputation ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "None — all columns complete!")
print(f"\nFinal shape: {df.shape}")
df.describe()

In [ ]:
df.to_csv('gurgaon_properties_missing_value_imputation.csv', index=False)
print("Saved gurgaon_properties_missing_value_imputation.csv")